# 🛣️ Post-Training: Evaluate, Export, Download

Jalankan cell ini SETELAH training selesai.

In [ ]:
import os
from ultralytics import YOLO
import torch

# Find best.pt automatically
BEST_PT = None
for root, dirs, files in os.walk("/content/runs"):
    for f in files:
        if f == "best.pt":
            BEST_PT = os.path.join(root, f)
            break
    if BEST_PT:
        break

if not BEST_PT:
    # Fallback: search everywhere
    for root, dirs, files in os.walk("/content"):
        for f in files:
            if f == "best.pt":
                BEST_PT = os.path.join(root, f)
                break
        if BEST_PT:
            break

print(f"Best model: {BEST_PT}")
print(f"Exists: {os.path.exists(BEST_PT) if BEST_PT else False}")

# Load model
model = YOLO(BEST_PT)
print("Model loaded!")

In [ ]:
# Evaluate
DATA_YAML = "/content/data.yaml"
print("Evaluating...")
metrics = model.val(data=DATA_YAML, imgsz=640, batch=16, device="0" if torch.cuda.is_available() else "cpu")
print(f"\nmAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

names = ["retak_memanjang", "pengelupasan_lapisan_permukaan", "lubang", "retak_kulit_buaya", "retak_blok", "retak_pinggir"]
print("\nPer-class:")
for i, (n, ap) in enumerate(zip(names, metrics.box.maps)):
    print(f"  {i}. {n}: {ap:.4f}")

In [ ]:
# Export TFLite
print("Exporting TFLite...")
model.export(format="tflite", imgsz=640, half=True, nms=True)

# Find exported file
weights_dir = os.path.dirname(BEST_PT)
for f in os.listdir(weights_dir):
    if f.endswith(".tflite"):
        tflite = os.path.join(weights_dir, f)
        size = os.path.getsize(tflite) / (1024*1024)
        print(f"TFLite: {tflite} ({size:.2f} MB)")
        break
else:
    print("TFLite not found in", weights_dir)

In [ ]:
# Download
import shutil
from google.colab import files

out = "jalanCerdas_model"
os.makedirs(out, exist_ok=True)

# Copy files
for f in ["best.pt", "best.tflite", "best.onnx"]:
    src = os.path.join(weights_dir, f)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(out, f))
        print(f"  {f} ({os.path.getsize(src)/1024/1024:.1f} MB)")

# Copy results
results = os.path.join(weights_dir, "..", "results.csv")
if os.path.exists(results):
    shutil.copy2(results, os.path.join(out, "results.csv"))
    print("  results.csv")

shutil.make_archive(out, 'zip', '.', out)
print(f"\nZipped: {out}.zip")
files.download(f"{out}.zip")

In [ ]:
# Test prediction
from IPython.display import Image, display

val_dir = "/content/dataset/val/images"
imgs = [f for f in os.listdir(val_dir) if f.endswith((".jpg",".png"))][:3]

for img in imgs:
    results = model.predict(source=os.path.join(val_dir, img), imgsz=640, conf=0.5, save=True, project="/content/runs", name="predict", exist_ok=True)
    print(f"\n{img}:")
    for r in results:
        for box in r.boxes:
            print(f"  {model.names[int(box.cls[0])]}: {float(box.conf[0]):.0%}")

# Show last prediction
preds = [f for f in os.listdir("/content/runs/predict") if f.endswith((".jpg",".png"))]
if preds:
    display(Image(f"/content/runs/predict/{preds[-1]}", width=600))